In [1]:
!pip install pycocotools ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 25.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics]


In [2]:
import random
import sys
import numpy as np
import torch

from functools import partial
from pathlib import Path
from pycocotools.coco import COCO
from torch.utils.data import DataLoader

PROJECT_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
COMPETITION = "multiple-object-video-detection-hw-2-ucu-ai-2026"

DIRECTORY = INPUT_ROOT / COMPETITION
if not (DIRECTORY / "train" / "annotations_coco.json").is_file():
    data_roots = sorted({path.parent.parent
                         for path in INPUT_ROOT.rglob("annotations_coco.json")
                         if path.parent.name == "train"})
    if len(data_roots) != 1:
        raise FileNotFoundError(
            "Attach the extracted competition data, or set DIRECTORY to its input folder."
        )
    DIRECTORY = data_roots[0]

INFO_TRAIN = DIRECTORY / "train" / "annotations_coco.json"
INFO_VAL = DIRECTORY / "val" / "annotations_coco.json"
INFO_TEST = DIRECTORY / "test_images.csv"

required_scripts = {
    "data_prep.py",
    "data_cv.py",
    "dataset_helpers.py",
    "torch_model.py",
    "yolo26_model.py",
    "experiment_runner.py",
    "experiment_results.py",
}

script_dirs = sorted({path.parent for path in INPUT_ROOT.rglob("torch_model.py")
    if path.parent.name == "scripts" and all((path.parent / filename).is_file()
                                              for filename in required_scripts)
})
scripts_parent = str(script_dirs[0].parent)

if scripts_parent not in sys.path:
    sys.path.insert(0, scripts_parent)

from scripts.data_prep import load_annotations
from scripts.model_builders import build_frcnn_model
from scripts.dataset_helpers import DetectionDataset, collate_data
from scripts.experiment_runner import run_torch_final
from scripts.experiment_results import model_training_results_final

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
EPOCHS = 8
FRAME_STEP = 5
BATCH_SIZE = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [4]:
coco_train = COCO(str(INFO_TRAIN))
coco_val = COCO(str(INFO_VAL))

train_frames, train_boxes = load_annotations(coco_train, split="train")
valid_frames, valid_boxes = load_annotations(coco_val, split="val")

train_annotations_by_image = {image_id: group for image_id, group in train_boxes.groupby("image_id")}
valid_annotations_by_image = {image_id: group for image_id, group in valid_boxes.groupby("image_id")}
sampled_train_frames = (train_frames.sort_values(["video", "frame_id"]).groupby("video", group_keys=False).nth[::FRAME_STEP].reset_index(drop=True))

loading annotations into memory...
Done (t=6.23s)
creating index...
index created!
loading annotations into memory...
Done (t=0.68s)
creating index...
index created!


In [5]:
train_dataset = DetectionDataset(frames=sampled_train_frames, directory=DIRECTORY,
                                 annotations_by_image=train_annotations_by_image,
                                 empty_annotations=train_boxes.iloc[:0].copy(), validation=False)
valid_dataset = DetectionDataset(frames=valid_frames, directory=DIRECTORY,
                                 annotations_by_image=valid_annotations_by_image,
                                 empty_annotations=valid_boxes.iloc[:0].copy(), validation=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, collate_fn=collate_data)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, collate_fn=collate_data)

In [6]:
output_root = PROJECT_ROOT / "outputs" / "frcnn_final"
train_kwargs = {
    "train_loader": train_loader,
    "valid_loader": valid_loader,
    "annotations_path": INFO_VAL,
    "device": device,
    "seed": SEED,
    "epochs": EPOCHS,
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "lr_scheduler": True,
    "step_size": 4
}

In [7]:
final_history, final_results = run_torch_final(**train_kwargs, model_method=build_frcnn_model, output_dir=output_root)

loading annotations into memory...
Done (t=0.46s)
creating index...
index created!
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 215MB/s]


Loading and preparing results...
DONE (t=0.31s)
creating index...
index created!
Inference time: 564.08 s
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=71.69s).
Accumulating evaluation results...
DONE (t=3.10s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.150
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.313
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.127
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.056
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.128
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.307
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.104
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.225
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.272
 Average Recall     (AR) @[ IoU=0.50:0.95 

In [8]:
final_results_wrap = model_training_results_final(output_root)

Training history


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,1.097985,0.0001,0.150203,0.313311,0.055598
1,2,0.910732,0.0001,0.158415,0.336285,0.061613
2,3,0.840680,0.0001,0.157568,0.329739,0.069841



Best validation results — epoch 2


,mAP,AP50,AP_small
1,0.158415,0.336285,0.061613
